Author: Enikő Gergely <br>
Date: October 2026
    
# Back-Gluteal line vs OBP Analysis

The aim for this notebook is to collect data for further analysis of the realtionship between the Back-Gluteal line and player performance, mainly OBP. I analyize 70 hitter batting stance in 2025 and their batting performance to have a better understanding of the relationship.

Hitter data is downloaded from Baseball Savant and includes player information and batting statistics. I run the automated posture data collection for the first 70 hitters in the data. Videos are downloaded from Baseball Savant for 2025 for all hitters. Then frames are cut from the video and hitter is cropped from the image. Cropped images are saved to a folder including player name and year. 

Cropped images are manually checked to remove any incorrect images. In the last section, I draw a silhouette and Back-Gluteal line on the images, and calculate its angle. 

Results are saved to an excel file 'All_player_results.xlsx' which includes image name, game date, angle, player name and id, handedness and team id.

 # Packages

In [7]:
# import functions from baseballcv - download videos:
from baseballcv.functions.savant_scraper import BaseballSavVideoScraper

# import functions from baseballcv - frames
from baseballcv.datasets.processing.datasets_processor import DatasetProcessor

# import functions from baseballcv - crop hitter from frames
from baseballcv.functions.load_tools import LoadTools
from ultralytics import YOLO
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path


# back-gluteal line
import requests
import pandas as pd


# Inputs

In [9]:
# DO NOT CHANGE:
year = 2025
start_dt= str(year) +'-04-01' # 04-01 are used for yearly data
end_dt = str(year) +'-09-15'   # 09-15 are used for yearly data
pitch_type= 'FF'

# VIDEO INPUTS:
max_return_videos= 50 
max_videos_per_game= 1


# Initialize LoadTools class
load_tools = LoadTools()

model_weights = load_tools.load_model('phc_detector', model_type = 'YOLO')

model = YOLO(model_weights)


2026-10-28 12:29:29,107 - LoadTools - INFO - Model found at models/od/YOLO/pitcher_hitter_catcher_detector/model_weights/pitcher_hitter_catcher_detector_v4.pt


In [8]:
# load qualified player data downlaoded from Baseball Savant for 2025:
df_player_list = pd.read_csv('stats_with_team.csv')
df_player_list.head()

,Unnamed: 0,"last_name, first_name",player_id,year,pa,k_percent,bb_percent,slg_percent,on_base_percent,on_base_plus_slg,...,sweet_spot_percent,barrel_batted_rate,hard_hit_percent,avg_best_speed,avg_hyper_speed,whiff_percent,swing_percent,team_id,team_name,team_abbreviation
0,0,"Torres, Gleyber",650402,2025,628,16.1,13.5,0.387,0.358,0.745,...,38.6,8.2,39.0,99.951682,94.284675,19.5,40.8,116,Detroit Tigers,DET
1,1,"Contreras, William",661388,2025,659,18.2,12.7,0.399,0.355,0.754,...,31.6,6.4,48.5,102.570622,95.839705,24.2,46.5,158,Milwaukee Brewers,MIL
2,2,"Cronenworth, Jake",630105,2025,515,20.8,13.4,0.377,0.367,0.744,...,31.9,6.2,39.0,99.154485,93.790990,18.8,42.7,135,San Diego Padres,SD
3,3,"Walker, Christian",572233,2025,640,27.7,6.3,0.421,0.297,0.718,...,34.5,12.9,46.1,102.499786,95.668558,30.5,52.2,117,Houston Astros,HOU
4,4,"McCutchen, Andrew",457705,2025,551,21.4,12.2,0.367,0.333,0.700,...,34.5,8.4,42.5,100.124243,94.328515,26.2,45.6,134,Pittsburgh Pirates,PIT


# Functions:

In [10]:

# Function to get game date using game_pk:
def get_date_from_game_pk(game_pk:int):

    """
    Gets the game date from the MLB Stats API using the game ID.

    Args:
        game_pk (int): The unique ID of the MLB game.

    Returns:
        str: The game date in YYYY-MM-DD format.
    """
    
    url = f"https://statsapi.mlb.com/api/v1/schedule?gamePk={game_pk}"
    data = requests.get(url).json()

    game_date = data["dates"][0]["games"][0]["gameDate"]

    return game_date[:10]



def get_batting_hand(player_id: int):
    """
    Gets a player's batting handedness from the MLB Stats API.

    Args:
        player_id (int): The MLB player ID.

    Returns:
        str: 'L' for left-handed, 'R' for right-handed,
             or 'S' for a switch hitter.
    """
    url = f"https://statsapi.mlb.com/api/v1/people/{player_id}"
    data = requests.get(url).json()

    return data["people"][0]["batSide"]["code"]


# Function to get back-gluteal line and its angle:
def back_gluteal_line(image_path: Path, output_dir: Path, handedness: str):
    """
        Detects the batter's silhouette, identifies the Back-Gluteal line, calculates its angle,
        and saves the annotated image to the selected output folder.

        Args:
            image_path (Path): Path to the input image of the batter.
            output_dir(Path): Directory where the annotated image will be saved.
            handedness (str): Batter's handedness ('R' for righ-handed, 'L' for 
            left-handed). If 'R', the image is flipped horizontally before processing.

        Returns:
            dict: A dictionary containing the input file name, calculated Back-Gluteal angle in degrees,
                and game date.
            None: If the image cannot be loaded or a silhouette or contour cannot be found.
    """

    # --------------------------------------------------
    # Load image
    # --------------------------------------------------

    # get game date:
    image_name = Path(image_path).name
    game_date = get_date_from_game_pk(image_name[14:20])
    print ("Game date: ", game_date)

    # load image:
    image = cv2.imread(str(image_path))

    if image is None:
        print(f"[ERROR] Cannot read: {image_path}")
        return None

    # if 'R', flip horizontally:
    if handedness == 'R':
        image = cv2.flip(image, 1)

    # image shape:
    print(f"Image shape: {image.shape}")

    h, w, _ = image.shape

    # --------------------------------------------------
    # YOLO segmentation and find best mask
    # --------------------------------------------------
    
    model = YOLO("yolov8n-seg.pt")
    
    results = model(image)[0]

    if results.masks is None:
        print("[ERROR] No masks detected")
        return None

    # get 40% of the width and half of height - this point should be part of the silhouette
    image_center = np.array([int(w * 0.4), int(h / 2)])
    print("Selected point to be part of the silhouette:", image_center)

    # select mask which includes the selected point and has the largest condifence:
    best_mask = None
    best_confidence = 0

    for mask, box in zip(results.masks.data, results.boxes):

        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        # Only person masks
        if class_id != 0:
            continue

        print(f"Person detected | confidence={confidence:.2f}")

        # --------------------------------------------------
        # Resize mask to the original image size
        # --------------------------------------------------

        mask_np = mask.cpu().numpy()

        mask_resized = cv2.resize(
            mask_np,
            (w, h),
            interpolation=cv2.INTER_NEAREST
        )

        # binary mask:
        mask_binary = mask_resized > 0.5

        # --------------------------------------------------
        # Is the selected point in the mask?
        # --------------------------------------------------

        x = image_center[0]
        y = image_center[1]

        point_inside = mask_binary[y, x]

        print(f"Point inside mask: {point_inside}")

        # If not, drop this mask:
        if not point_inside:
            continue

        # --------------------------------------------------
        # If point is included, select the mask with the largest confidence.
        # --------------------------------------------------

        if confidence > best_confidence:

            best_confidence = confidence
            best_mask = mask_binary


    # --------------------------------------------------
    # Check
    # --------------------------------------------------

    if best_mask is None:

        print("[ERROR] No person mask contains the selected point")
        return None

    print(
        f"Selected person | confidence={best_confidence:.2f}"
    )

    # --------------------------------------------------
    # Find contour
    # --------------------------------------------------

    mask_binary = best_mask.astype(np.uint8) * 255

    kernel = np.ones((15, 15), np.uint8)

    # connect all countour parts to have only one contour:
    mask_binary = cv2.morphologyEx(
        mask_binary.astype(np.uint8),
        cv2.MORPH_CLOSE,
        kernel
    )

    contours, _ = cv2.findContours(
        mask_binary,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours) == 0:
        print("[ERROR] No contour found")
        return None

    # select the contour with the largest area for player contour
    player_contour = max(
        contours,
        key=cv2.contourArea
    )

    print(
        "Selected silhouette area:",
        cv2.contourArea(player_contour)
    )


    # --------------------------------------------------
    # Draw silhouette
    # --------------------------------------------------

    cv2.drawContours(
        image,
        [player_contour],
        -1,
        (0, 255, 0),
        3
    )

    plt.figure(figsize=(12, 8))

    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

    plt.axis("off")
    plt.title("Detected batter silhouette")
    plt.show()

    # --------------------------------------------------
    # Largest countour
    # --------------------------------------------------
    
    # select the contour with the largest area
    largest_contour = 0
    largest_area = 0

    for i, contour in enumerate(contours):

        area = cv2.contourArea(contour)

        x, y, w_c, h_c = cv2.boundingRect(contour)

        print(
            f"Contour {i}: "
            f"area={area:.0f}, "
            f"x={x}, y={y}, "
            f"w={w_c}, h={h_c}"
        )

        if area > largest_area:
            largest_area = area
            largest_contour = i

    print ("Selected contour = ", i)


    # --------------------------------------------------
    # Find Back point
    # --------------------------------------------------

    # get the top 5-30% of heigth of the image
    y_Back_bottom = int(h *0.3 )
    print("Selected point:", y_Back_bottom)

    y_Back_top = int(h *0.05 )
    print("Selected point:", y_Back_top)


    # Filter contour pairs where y_Back_bottom > y > y_Back_top 
    contour = contours[largest_contour].reshape(-1, 2)
    filtered = [[[x, y]] for x, y in contour if y < y_Back_bottom  and y > y_Back_top]
    filtered_array = np.array(filtered)

    # find the smallest x's index:
    min_x_idx = np.argmin(filtered_array[:, 0, 0])

    # the index's x and y:
    back_point_x = filtered_array[min_x_idx, 0, 0]
    back_point_y = filtered_array[min_x_idx, 0, 1]

    print(f"Back point X: {back_point_x}, and its Y: {back_point_y}")

    # --------------------------------------------------
    # Find Gluteal point
    # --------------------------------------------------

    # get the top 30-60% of heigth of the image
    y_Gluteal_bottom = int(h *0.6 )
    print("Selected point:", y_Gluteal_bottom)

    # Filter pairs where y_Gluteal_bottom > y > y_Back_bottom
    filtered = [[[x, y]] for x, y in contour if y > y_Back_bottom and y < y_Gluteal_bottom]
    filtered_array = np.array(filtered)

    # find the smallest x's index:
    min_x_idx = np.argmin(filtered_array[:, 0, 0])

    # the index's x and y:
    gluteal_point_x = filtered_array[min_x_idx, 0, 0]
    gluteal_point_y = filtered_array[min_x_idx, 0, 1]

    print(f"Gluteal point X: {gluteal_point_x}, and its Y: {gluteal_point_y}")

    #filtered_player_contour = filtered_array.reshape(-1, 1, 2)

    # --------------------------------------------------
    # Draw the Back-Gluteal Line and horizontal segmentation lines
    # --------------------------------------------------

    P1 = np.array([back_point_x, back_point_y])  # back
    P2 = np.array([gluteal_point_x, gluteal_point_y])  # glutes


    # Draw a line on the two points
    dx = P2[0] - P1[0]
    dy = P2[1] - P1[1]

    length = np.sqrt(dx**2 + dy**2)

    ux = dx / length
    uy = dy / length

    # extension of the line
    extension = 50

    A = P1 - extension * np.array([ux, uy])
    B = P2 + extension * np.array([ux, uy])


    cv2.line(
        image,
        tuple(A.astype(int)),
        tuple(B.astype(int)),
        (0, 0, 255),
        3
    )

    # add horizontal lines for the segmentations:
    # Bottom of the Gluteal section
    cv2.line(
        image,
        (0, y_Gluteal_bottom),
        (w, y_Gluteal_bottom),
        (255, 0, 0),
        2
    )
    # Bottom of the Back section/ Top of the Gluteal section
    cv2.line(
        image,
        (0, y_Back_bottom),
        (w, y_Back_bottom),
        (255, 0, 0),
        2
    )
    # Top of the Back section
    cv2.line(
        image,
        (0, y_Back_top),
        (w, y_Back_top),
        (255, 0, 0),
        2
    )
    # --------------------------------------------------
    # Calculate the angle in degrees
    # --------------------------------------------------

    angle = np.degrees(np.arctan2(dy, dx))

    print(f"Angle: {angle:.2f}°")
    
    # --------------------------------------------------
    # Display the image
    # --------------------------------------------------

   # if 'R', flip back horizontally:
    if handedness == 'R':
        image = cv2.flip(image, 1)

    plt.figure(figsize=(12, 8))

    plt.imshow(
        cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    )

    plt.axis("off")
    plt.title("Detected body contour")
    plt.show()


    # --------------------------------------------------
    # Save final image
    # --------------------------------------------------

    output_dir.mkdir(parents=True, exist_ok=True)

    out_path = output_dir / f"{game_date}_{image_name}"

    cv2.imwrite(str(out_path), image)
    print(f"[OK] -> {out_path}")

    # --------------------------------------------------
    # Return angle data
    # --------------------------------------------------
    
    return {"file_name": image_path.name, "angle": angle, "Date": game_date}

# Automated Data Collection:

This part will download savant videos, cut out frames and crop hitters and save them to their own player folder.

For the analysis I only ran the process for the first 70 players which is already enough to have a better understanding of the relationship between the Back-Gluteal angle and player performance.

In [ ]:
#Inputs:
for i in range(0, 70): #len(df)):
    
    print(i)
    
    # PLAYER INPUTS:
    player= df_player_list['player_id'][i] 
    player_name = df_player_list['last_name, first_name'][i]
    team_abbr= df_player_list['team_abbreviation'][i]

    # videos folder:
    videos_download_folder= Path(f"{player_name}({player})") / f"savant_videos_{player_name}_{year}"

    # frames and cropped images:
    output_frames_folder = Path(f"{player_name}({player})") / f"hitting_frames_{player_name}_{year}"
    cropped_images_output_folder = Path(f"{player_name}({player})") /f"cropped_images_{player_name}_{year}"


    #----------------------------------------
    # Download Videos:
    #----------------------------------------
    scraper = BaseballSavVideoScraper.from_date_range(start_dt= start_dt, 
                                                      end_dt = end_dt,  
                                                      team_abbr= team_abbr,
                                                      player= player,  
                                                      pitch_type= pitch_type, 
                                                      download_folder= videos_download_folder, 
                                                      max_return_videos= max_return_videos,  
                                                      max_videos_per_game= max_videos_per_game)

    #load pitch level information into a dataframe
    df = scraper.play_ids_df

    # Filter for home games only:
    df = df[df['home_team'] == team_abbr]

    # Pitch level information:
    print("Number of pitches availbale:", len(df))

    # Save final df to excel:
    df.to_excel( Path(f"{player_name}({player})")/f"savant_videos_{player_name}_{year}.xlsx" , index=False)
    
    # run scraper to download videos (with the name of game_pk + game_id):
    scraper.play_ids_df = df
    scraper.run_executor()
    
    #----------------------------------------
    # Cut out frames:
    #----------------------------------------
    processor = DatasetProcessor()
    processor.generate_photo_dataset( video_folder = videos_download_folder, 
                                  output_frames_folder =  output_frames_folder,
                                  max_num_frames = 1,
                                  frame_stride = 100)
    
    
    #----------------------------------------
    # Crop hitter:
    #----------------------------------------
    
    for file_name in os.listdir(output_frames_folder):
        if file_name.lower().endswith(".png"):

            #IMAGE PATH:
            SOURCE_IMAGE_PATH = output_frames_folder / file_name
            pure_name = os.path.splitext(file_name)[0]
            print("Source Image Updated to", file_name)

            # Box Detection with Pre-Trained Pitcher/Hitter/Catcher Detection Model
            results = model.predict(source=SOURCE_IMAGE_PATH, save=True)

            # Process Box Detections:
            print("Return Box Predictions as Tensors:")

            for r in results: #access predictions from frame
                for box in r.boxes.cpu().numpy(): #access individual boxes from frame, move to cpu memory, convert to numpy array from tensor
                    print(box.xywh) #print x, y coordinate and width/height of box as tensor
                    print(f"Class Value: {box.cls} \n")

            # Utilizing OpenCV to Visualize Predictions
            # Read the image:
            image = cv2.imread(SOURCE_IMAGE_PATH)

            # Convert image to RGB for correct matplotlib display
            image_rgb = cv2.cvtColor(image.copy(), cv2.COLOR_BGR2RGB)

            # Output folder for cropped images
            os.makedirs(cropped_images_output_folder, exist_ok=True)  # Create folder if not exists

            # Loop through all detected boxes and fint he batter: box.cls = 0
            for box in r.boxes.cpu().numpy():
                if box.cls == 0:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])  # Convert to integer

                    # Draw rectangle/box:
                    cv2.rectangle(image_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)

                    # Crop and save it to folder
                    cropped_img = image[y1:y2, x1:x2]

                    save_path = os.path.join(cropped_images_output_folder, "cropped_image_"+ pure_name +".jpg")
                    cv2.imwrite(save_path, cropped_img)

                    print(f"Cropped image saved at: {save_path}")

            # Show the final image with hitter rectangle
            plt.imshow(image_rgb)
            plt.axis('off')
            plt.show()

# Manual data check
At this point, it worth to do a quick look at the cropped images, to make sure data quality is good and delete incorrect images.


# Back-Gluteal line and Anlge Analysis
This code will go through the first 70 hitters, draw a silhouette and Back-Gluteal line on the images, and calculate its angle. Results are saved to an excel file 'All_player_results.xlsx' which includes image name, game date, angle, player name and id, handedness and team id. 

In [ ]:
#Inputs:

all_player_output_results = []

for i in range(0, 70): #len(df)):
    
    # PLAYER INPUTS:
    player= df_player_list['player_id'][i] 
    player_name = df_player_list['last_name, first_name'][i]
    team_abbr= df_player_list['team_abbreviation'][i]

    # videos folder:
    videos_download_folder= Path(f"{player_name}({player})") / f"savant_videos_{player_name}_{year}"

    # frames and cropped images:
    output_frames_folder = Path(f"{player_name}({player})") / f"hitting_frames_{player_name}_{year}"
    cropped_images_output_folder = Path(f"{player_name}({player})") /f"cropped_images_{player_name}_{year}"
    
    #------------------------------
    # Handedness
    #------------------------------
    handedness = get_batting_hand(player)
    
    print(f"Batter handedness: {handedness}")
    
    if handedness == 'S':
        continue
    #------------------------------
    # Run Analysis
    #------------------------------
    
    # create empty list for results:
    all_results = []

    # folder where cropped images are saved
    cropped_images_input_dir = Path(f"{player_name}({player})\cropped_images_{player_name}_{year}")

    # loop through all the files and calculate angle
    for file_name in os.listdir(cropped_images_input_dir):
        print(file_name)

        result = back_gluteal_line(cropped_images_input_dir /file_name, 
                                   Path(f"{player_name}({player})/angle_results_{player_name}_{year}"), 
                                   handedness)
            

        if result:
            # Add new items to results:
            result["player_name"] = player_name
            result["player_id"] = player
            result["handedness"] = handedness
            result["team"] = team_abbr
            all_results.append(result)

    # Create dataframe:
    all_results_df = pd.DataFrame(all_results) 
    
    # Save to excel
    all_results_df.to_excel( Path(f"{player_name}({player})")/f"Angle_data_{player_name}_{year}.xlsx" , index=False)
    print("Excel file has been saved to player folder.")

    
    #------------------------------
    # Final Output
    #------------------------------

    all_player_output_results.append(all_results)

# Create dataframe:
flat_data = [item for sublist in all_player_output_results for item in sublist]

all_player_output_results_df = pd.DataFrame(flat_data)

# Save to excel
all_player_output_results_df.to_excel( 'All_player_results.xlsx' , index=False)
print("Excel file has been saved to main folder.")

In [11]:
# final output:
all_player_output_results_df

,file_name,angle,Date,player_name,player_id,handedness,team
0,cropped_image_776642_c329fdbf-e52c-3506-9f1a-e...,97.028396,2025-08-21,"Bogaerts, Xander",593428,R,SD
1,cropped_image_776789_1ef77b34-1fd1-3612-aa31-0...,91.112400,2025-08-10,"Bogaerts, Xander",593428,R,SD
2,cropped_image_776807_048037ae-9da8-36c2-856f-0...,101.469530,2025-08-10,"Bogaerts, Xander",593428,R,SD
3,cropped_image_776896_27a00317-25e0-3004-afd7-5...,100.561011,2025-08-03,"Bogaerts, Xander",593428,R,SD
4,cropped_image_777203_ea34826d-2aec-3cfb-a4d7-1...,89.409343,2025-07-08,"Bogaerts, Xander",593428,R,SD
...,...,...,...,...,...,...,...
493,cropped_image_777046_0426362b-3496-3fb2-8b37-a...,104.036243,2025-07-22,"Clement, Ernie",676391,R,TOR
494,cropped_image_777283_7dcb6011-8bae-31bc-987b-f...,109.573126,2025-07-01,"Clement, Ernie",676391,R,TOR
495,cropped_image_777651_25a98f7b-1c3e-3443-ac9c-0...,111.540976,2025-06-04,"Clement, Ernie",676391,R,TOR
496,cropped_image_777816_ce20e954-1034-3bae-9531-8...,106.699244,2025-05-22,"Clement, Ernie",676391,R,TOR
